# Lesson 2: Database Modeling & SQL Mastery
### VisionStream — From CSV to Production PostgreSQL

---

## Learning Objectives

By the end of this lesson, you will be able to:

1. **Design** a relational database schema with proper relationships and constraints
2. **Write** DDL (Data Definition Language) SQL to create tables and indexes
3. **Explain** why indexes matter for query performance
4. **Write** a complex aggregation SQL query with GROUP BY and geographic filtering
5. **Import** hundreds of thousands of real GPS records into PostgreSQL using batch insert

---

## Why Not Just Use a CSV File?

| | CSV File | PostgreSQL |
|---|---|---|
| **Scale** | ~100k rows before slowdown | Billions of rows with proper indexes |
| **Queries** | Full file scan every time | Index-based lookup in milliseconds |
| **Concurrent writes** | Corrupts data | ACID transactions prevent corruption |
| **Relationships** | No enforcement | Foreign keys guarantee integrity |
| **Analytics** | Manual Python loops | SQL GROUP BY, aggregations, window functions |
| **Backup/Recovery** | Manual copy | Built-in WAL (Write-Ahead Log) |

For a system receiving 1,000 sensor readings per second from 1,000 helmets,
a CSV file would be overwritten and corrupted almost immediately.

## Section 1: Entity-Relationship Design

Before writing any SQL, draw the schema. Here is the VisionStream ER diagram:

```
┌─────────────────────────────────┐
│            devices              │
├─────────────────────────────────┤
│ id             UUID (PK)        │
│ device_id      VARCHAR UNIQUE   │◄──────┐
│ owner_name     VARCHAR          │       │ FK (1:N)
│ registered_at  TIMESTAMPTZ      │       │
│ firmware_version VARCHAR        │       │
│ is_active      BOOLEAN          │       │
└─────────────────────────────────┘       │
                                           │
        ┌──────────────────────────────────┤
        │                                  │
┌───────▼──────────────────────┐  ┌────────▼──────────────────────┐
│       telemetry_logs         │  │           alerts               │
├──────────────────────────────┤  ├───────────────────────────────┤
│ id          BIGSERIAL (PK)   │  │ id          BIGSERIAL (PK)    │
│ device_id   VARCHAR (FK)     │  │ device_id   VARCHAR (FK)      │
│ timestamp   TIMESTAMPTZ      │  │ timestamp   TIMESTAMPTZ       │
│ gps_lat     DOUBLE PRECISION │  │ alert_type  VARCHAR           │
│ gps_lon     DOUBLE PRECISION │  │ gps_lat     DOUBLE PRECISION  │
│ obstacle_cm INTEGER          │  │ gps_lon     DOUBLE PRECISION  │
│ alert_type  VARCHAR          │  │ severity    VARCHAR (CHECK)   │
│ battery     SMALLINT         │  │ resolved    BOOLEAN           │
└──────────────────────────────┘  └───────────────────────────────┘
```

### Relationships

- `devices` → `telemetry_logs` : **One-to-Many** (1 device, many log records)
- `devices` → `alerts`         : **One-to-Many** (1 device, many alert events)
- `device_id` is the foreign key that links these tables

### Key Design Decisions

| Decision | Reasoning |
|----------|----------|
| `UUID` for `devices.id` | Avoids primary key collisions when multiple API servers insert simultaneously (scale-out safety) |
| `BIGSERIAL` for logs/alerts | Supports up to 9.2×10¹⁸ rows; regular `SERIAL` maxes at ~2.1 billion |
| Separate `alerts` table | Telemetry logs are write-heavy; alert analytics are read-heavy. Separating them allows different optimization strategies |
| `TIMESTAMPTZ` not `TIMESTAMP` | Stores timezone-aware timestamps — critical when helmets are deployed globally |

## Section 2: Indexes — The Secret to Query Performance

An **index** is like a book's table of contents. Without it, PostgreSQL reads every row
to find matches (a *sequential scan*). With it, PostgreSQL jumps directly to the matching rows.

### Query Performance Comparison (hypothetical)

Query: Find all telemetry for device `helmet-001` in the last 24 hours.

| Without index | With composite index on `(device_id, timestamp DESC)` |
|---------------|-------------------------------------------------------|
| Scans all 500M rows | Skips directly to ~86,400 matching rows |
| ~30 seconds | ~5 milliseconds |
| Blocks other queries | Non-blocking |

### The Three Indexes You Need

```sql
-- Index 1: For GET /telemetry/{device_id}/history
-- Query pattern: WHERE device_id = ? ORDER BY timestamp DESC
CREATE INDEX idx_telemetry_device_time
    ON telemetry_logs (device_id, timestamp DESC);

-- Index 2: For GET /alerts/history?device_id=...
-- Query pattern: WHERE device_id = ? ORDER BY timestamp DESC
CREATE INDEX idx_alerts_device_time
    ON alerts (device_id, timestamp DESC);

-- Index 3: For GET /alerts/stats (the geo-grid query)
-- Query pattern: WHERE gps_lat BETWEEN ? AND ? AND gps_lon BETWEEN ? AND ?
CREATE INDEX idx_alerts_location
    ON alerts (gps_lat, gps_lon);
```

### When NOT to add an index
- Columns that are never filtered or sorted on
- Very small tables (< 1,000 rows) — sequential scan is faster
- Columns that are updated very frequently (indexes slow down writes)

## Section 3: Illustrative Code — SQL DDL Pattern

Study the pattern below. It shows the structure for `CREATE TABLE` with:
- Primary keys, foreign keys, and NOT NULL constraints
- A `CHECK` constraint that enforces data integrity at the database level
- `ON DELETE CASCADE` behavior

**Do NOT copy this directly.** Use it as a reference when writing `db/schema.sql`.

In [ ]:
# ── ILLUSTRATIVE SQL PATTERNS — Study, then write your own in db/schema.sql ──

# Pattern 1: Table with UUID primary key and server-side defaults
example_devices_ddl = """
CREATE TABLE IF NOT EXISTS example_table (
    id            UUID         PRIMARY KEY DEFAULT gen_random_uuid(),
    name          VARCHAR(64)  UNIQUE NOT NULL,
    created_at    TIMESTAMPTZ  NOT NULL DEFAULT NOW(),
    is_active     BOOLEAN      NOT NULL DEFAULT TRUE
);
"""

# Pattern 2: Table with BIGSERIAL PK, foreign key, and CHECK constraint
example_events_ddl = """
CREATE TABLE IF NOT EXISTS example_events (
    id          BIGSERIAL    PRIMARY KEY,
    table_id    VARCHAR(64)  NOT NULL
                REFERENCES example_table(name) ON DELETE CASCADE,
    event_type  VARCHAR(32)  NOT NULL,
    severity    VARCHAR(16)  NOT NULL
                CHECK (severity IN ('LOW', 'MEDIUM', 'HIGH')),
    timestamp   TIMESTAMPTZ  NOT NULL
);
"""

# Pattern 3: Composite index on two columns
example_index_ddl = """
CREATE INDEX IF NOT EXISTS idx_events_table_time
    ON example_events (table_id, timestamp DESC);
"""

print("Study these patterns, then fill in db/schema.sql")
print("Your schema needs: devices, telemetry_logs, alerts + 3 indexes")

## Section 4: The Complex Query — 24-Hour Alert Frequency by Geo-Grid

This is the most important query in the system. It powers the safety analytics dashboard.

### The Business Question

> *"In the past 24 hours, which locations on UC Berkeley campus had the most
> dangerous obstacle collisions?"*

### Geographic Grid Concept

GPS coordinates have too much precision for heatmap grouping:
- `37.871945` and `37.871952` are only 0.7 meters apart — they're the same location for our purposes

Solution: **Round coordinates to 2 decimal places** (≈ 1.1km resolution):
```
ROUND(gps_lat::numeric, 2) → 37.87   ← all points in the same ~1.1km cell
ROUND(gps_lon::numeric, 2) → -122.26
```

### Query Structure (annotated)

```sql
SELECT
    ROUND(gps_lat::numeric, 2)  AS geo_grid_lat,   -- Grid cell lat
    ROUND(gps_lon::numeric, 2)  AS geo_grid_lon,   -- Grid cell lon
    alert_type,
    COUNT(*)                    AS alert_count      -- Events in this cell
FROM alerts
WHERE
    timestamp  > NOW() - INTERVAL '24 hours'        -- Time window
    AND gps_lat BETWEEN 37.85 AND 37.90             -- Berkeley campus lat
    AND gps_lon BETWEEN -122.28 AND -122.22         -- Berkeley campus lon
    AND severity IN ('HIGH', 'CRITICAL')            -- Significant events only
GROUP BY
    ROUND(gps_lat::numeric, 2),
    ROUND(gps_lon::numeric, 2),
    alert_type
ORDER BY alert_count DESC;
```

### Parameterized version (for use with SQLAlchemy `text()`)

In `app/services/alert_service.py`, replace hardcoded values with named parameters:
```sql
WHERE timestamp > :time_window_start
  AND gps_lat BETWEEN :lat_min AND :lat_max
  AND gps_lon BETWEEN :lon_min AND :lon_max
```
Pass them as a dict: `db.execute(sql, {"time_window_start": ..., "lat_min": ..., ...})`

## Section 5: The Microsoft Geolife Dataset

For Lesson 2, you will import **real GPS trajectory data** into your database.
The seed script will populate **all three tables**: `devices`, `telemetry_logs`, and `alerts`.

### Dataset Information

| Property | Value |
|----------|-------|
| **Name** | Microsoft Research Asia — Geolife GPS Trajectories |
| **Download** | https://www.microsoft.com/en-us/research/project/geolife-building-social-networks-using-human-location-history/ |
| **Size** | ~298 MB compressed, 17,621 trajectories, 182 users |
| **Coverage** | Primarily Beijing, China (2007–2012) |

### .plt File Format

Each file contains one GPS trajectory. Lines 1–6 are headers (ignore them).
Line 7 onward is data, comma-separated:

```
Geolife GPS track
WGS 84
Altitude is in Feet
Reserved 3
0,2,255,My Tracks,0,0,2,8421376
0
39.906631,116.385564,0,492,40097.5864583333,2009-10-11,14:04:29   ← data starts here
39.906611,116.385544,0,492,40097.5865277778,2009-10-11,14:04:36
```

Column layout (0-indexed):
- `[0]` Latitude
- `[1]` Longitude
- `[2]` Always 0 (ignore)
- `[3]` Altitude in feet (ignore for VisionStream)
- `[4]` Fractional days since 12/30/1899 (ignore — use date+time columns instead)
- `[5]` Date string: `YYYY-MM-DD`
- `[6]` Time string: `HH:MM:SS`

### What the .plt file provides vs. what you must generate

The .plt files only contain GPS coordinates and timestamps. All other fields
must be **generated with random values** in your seed script:

| Table | Column | Source |
|-------|--------|--------|
| `devices` | `device_id` | **You generate** — e.g., `"geolife-seed-device-001"` |
| `devices` | `owner_name` | **You generate** — e.g., `"Geolife Seed"` |
| `devices` | `firmware_version` | **You generate** — e.g., `"v1.0-seed"` |
| `devices` | `is_active` | **You set** — `True` |
| `telemetry_logs` | `gps_lat`, `gps_lon` | From .plt file |
| `telemetry_logs` | `timestamp` | From .plt file |
| `telemetry_logs` | `obstacle_distance_cm` | **You generate** — random integer 0–500 |
| `telemetry_logs` | `alert_type` | **You generate** — `"NONE"` for most, occasionally pick a real alert type |
| `telemetry_logs` | `battery_level` | **You generate** — random integer 0–100 |
| `alerts` | `gps_lat`, `gps_lon` | From .plt file (same as telemetry) |
| `alerts` | `timestamp` | From .plt file (same as telemetry) |
| `alerts` | `alert_type` | **You generate** — pick from real alert types |
| `alerts` | `severity` | **You generate** — pick from `'LOW'`, `'MEDIUM'`, `'HIGH'`, `'CRITICAL'` |
| `alerts` | `resolved` | **You generate** — random `True`/`False` |

> **Important:** `obstacle_distance_cm` and `battery_level` must NOT be `None`.
> The seed data should simulate real sensor readings, so generate plausible random values.

### Insertion Order (Critical!)

Because of **foreign key constraints**, you must insert data in this exact order:

```
Step 1: INSERT INTO devices          ← Parent table (no dependencies)
Step 2: INSERT INTO telemetry_logs   ← Child table (depends on devices.device_id)
Step 3: INSERT INTO alerts           ← Child table (depends on devices.device_id)
```

If you try to insert telemetry_logs or alerts before the device exists,
PostgreSQL will throw a **foreign key violation error**.

### Download and Setup Steps

```bash
# 1. Download from the URL above (look for the .zip download link)
# 2. Extract to: VisionStream/data/geolife/
# 3. Directory structure should be:
#      data/geolife/Data/000/Trajectory/20081023025304.plt
#      data/geolife/Data/001/Trajectory/...
# 4. Run: python scripts/seed_geolife_data.py
```

## Section 6: Illustrative Code — Batch Insert Pattern for All Three Tables

Importing 100,000+ rows requires batch insertion.
**Never** insert one row at a time in a loop — that's 100,000 round-trips to the database.

The pattern below shows how to seed all three tables in the correct order,
using `bulk_insert_mappings()` to send all rows in a single SQL statement.

In [ ]:
# ── ILLUSTRATIVE EXAMPLE ONLY — Study the pattern ──
#
# This shows the complete seed flow for all 3 tables:
#   1. Insert a device (parent)
#   2. Parse .plt files → insert telemetry_logs with random sensor data
#   3. Generate alerts from a subset of telemetry records

# from app.database import SessionLocal
# from app.models.device import Device
# from app.models.telemetry import TelemetryLog
# from app.models.alert import Alert
# from datetime import datetime
# import random

# ── Step 1: Insert a device first (parent table) ──
#
# The device must exist BEFORE you insert telemetry_logs or alerts
# because both child tables have a FOREIGN KEY referencing devices.device_id.
#
# SEED_DEVICE_ID = "geolife-seed-device-001"
#
# def insert_seed_device() -> None:
#     """Insert one placeholder device so foreign keys are valid."""
#     db = SessionLocal()
#     try:
#         existing = db.query(Device).filter(
#             Device.device_id == SEED_DEVICE_ID
#         ).first()
#         if not existing:
#             device = Device(
#                 device_id=SEED_DEVICE_ID,
#                 owner_name="Geolife Seed Data",
#                 firmware_version="v1.0-seed",
#                 is_active=True,
#             )
#             db.add(device)
#             db.commit()
#             print(f"Inserted device: {SEED_DEVICE_ID}")
#         else:
#             print(f"Device {SEED_DEVICE_ID} already exists, skipping.")
#     finally:
#         db.close()


# ── Step 2: Parse .plt file and generate random sensor data ──
#
# The .plt file only provides GPS + timestamp.
# You must generate random values for the missing fields.
#
# def parse_plt_file(filepath: str) -> list[dict]:
#     """Parse a .plt file and return telemetry dicts with random sensor data."""
#     records = []
#     with open(filepath, "r") as f:
#         for i, line in enumerate(f):
#             if i < 6:          # Skip header lines
#                 continue
#             parts = line.strip().split(",")
#             if len(parts) < 7:
#                 continue
#
#             gps_lat = float(parts[0])
#             gps_lon = float(parts[1])
#             date_str = parts[5]
#             time_str = parts[6]
#             timestamp = datetime.strptime(
#                 f"{date_str} {time_str}", "%Y-%m-%d %H:%M:%S"
#             )
#
#             # Generate random sensor data (NOT None!)
#             obstacle_cm = random.randint(0, 500)
#             battery = random.randint(10, 100)
#
#             # Occasionally generate a non-NONE alert type
#             # (about 5% of records will have an alert)
#             if random.random() < 0.05:
#                 alert_type = random.choice([
#                     "OBSTACLE_WARNING",
#                     "OBSTACLE_CRITICAL",
#                     "FALL_DETECTED",
#                     "CRITICAL_COLLISION",
#                 ])
#             else:
#                 alert_type = "NONE"
#
#             records.append({
#                 "device_id":            SEED_DEVICE_ID,
#                 "timestamp":            timestamp,
#                 "gps_lat":              gps_lat,
#                 "gps_lon":              gps_lon,
#                 "obstacle_distance_cm": obstacle_cm,  # NOT None
#                 "alert_type":           alert_type,
#                 "battery_level":        battery,       # NOT None
#             })
#     return records


# ── Step 3: Batch insert telemetry_logs ──
#
# def batch_insert_telemetry(records: list[dict]) -> int:
#     """Bulk insert telemetry records in a single SQL statement."""
#     db = SessionLocal()
#     try:
#         db.bulk_insert_mappings(TelemetryLog, records)
#         db.commit()
#         return len(records)
#     except Exception as e:
#         db.rollback()
#         raise e
#     finally:
#         db.close()


# ── Step 4: Generate alerts from telemetry records ──
#
# Only telemetry records with alert_type != "NONE" become alerts.
# This simulates the real system behavior where the helmet detects
# an obstacle and triggers a safety alert.
#
# def generate_alerts_from_telemetry(telemetry_records: list[dict]) -> list[dict]:
#     """Filter telemetry records with alerts and convert to alert dicts."""
#     severity_map = {
#         "OBSTACLE_WARNING":  "LOW",
#         "OBSTACLE_CRITICAL": "HIGH",
#         "FALL_DETECTED":     "HIGH",
#         "CRITICAL_COLLISION": "CRITICAL",
#     }
#
#     alerts = []
#     for rec in telemetry_records:
#         if rec["alert_type"] != "NONE":
#             alerts.append({
#                 "device_id":  rec["device_id"],
#                 "timestamp":  rec["timestamp"],
#                 "alert_type": rec["alert_type"],
#                 "gps_lat":    rec["gps_lat"],
#                 "gps_lon":    rec["gps_lon"],
#                 "severity":   severity_map.get(rec["alert_type"], "LOW"),
#                 "resolved":   random.choice([True, False]),
#             })
#     return alerts


# ── Step 5: Batch insert alerts ──
#
# def batch_insert_alerts(records: list[dict]) -> int:
#     """Bulk insert alert records in a single SQL statement."""
#     db = SessionLocal()
#     try:
#         db.bulk_insert_mappings(Alert, records)
#         db.commit()
#         return len(records)
#     except Exception as e:
#         db.rollback()
#         raise e
#     finally:
#         db.close()


# ── Step 6: Orchestrate the full import ──
#
# def seed_database(max_files: int = 50) -> None:
#     """Orchestrate the full seed: device → telemetry → alerts."""
#     # Step 1: Insert device first
#     insert_seed_device()
#
#     # Step 2: Find all .plt files
#     import glob
#     pattern = os.path.join(GEOLIFE_DATA_DIR, "**", "*.plt")
#     all_files = glob.glob(pattern, recursive=True)
#     files_to_process = all_files[:max_files] if max_files else all_files
#
#     print(f"Found {len(all_files)} .plt files. Processing {len(files_to_process)}.")
#
#     # Step 3: Parse and batch-insert telemetry
#     batch = []
#     all_telemetry_records = []
#     total_inserted = 0
#
#     for i, filepath in enumerate(files_to_process):
#         records = parse_plt_file(filepath)
#         all_telemetry_records.extend(records)
#         batch.extend(records)
#
#         if len(batch) >= BATCH_SIZE:
#             inserted = batch_insert_telemetry(batch)
#             total_inserted += inserted
#             batch = []
#
#         if (i + 1) % 10 == 0:
#             print(f"  Parsed {i + 1}/{len(files_to_process)} files, "
#                   f"{total_inserted} telemetry rows inserted...")
#
#     # Insert remaining telemetry records
#     if batch:
#         total_inserted += batch_insert_telemetry(batch)
#
#     print(f"Telemetry done: {total_inserted} rows inserted.")
#
#     # Step 4: Generate and insert alerts
#     alert_records = generate_alerts_from_telemetry(all_telemetry_records)
#     if alert_records:
#         # Insert alerts in batches too
#         alert_batch = []
#         total_alerts = 0
#         for alert in alert_records:
#             alert_batch.append(alert)
#             if len(alert_batch) >= BATCH_SIZE:
#                 total_alerts += batch_insert_alerts(alert_batch)
#                 alert_batch = []
#         if alert_batch:
#             total_alerts += batch_insert_alerts(alert_batch)
#         print(f"Alerts done: {total_alerts} rows inserted.")
#     else:
#         print("No alerts generated (no non-NONE alert types found).")
#
#     print(f"Seed complete! Total: {total_inserted} telemetry + {total_alerts} alerts.")


# Performance comparison:
comparison = {
    "One INSERT per row (loop)": "100,000 round-trips to DB ≈ 5 minutes",
    "bulk_insert_mappings() (batch 1000)": "100 round-trips to DB ≈ 3 seconds",
    "COPY command (psql)": "1 command ≈ 0.5 seconds (fastest, but requires direct DB access)",
}

for method, speed in comparison.items():
    print(f"{method:45s} → {speed}")

---

## 📌 YOUR TASK — File Map

### Step 1 — Write the Database Schema (`db/schema.sql`)

Open `db/schema.sql` and implement all three `CREATE TABLE` statements:

| Table | Key Columns | Constraints |
|-------|-------------|-------------|
| `devices` | id (UUID PK), device_id (UNIQUE), owner_name, registered_at, is_active | gen_random_uuid(), DEFAULT NOW() |
| `telemetry_logs` | id (BIGSERIAL PK), device_id (FK), timestamp, gps_lat, gps_lon, alert_type | FK ON DELETE CASCADE |
| `alerts` | id (BIGSERIAL PK), device_id (FK), timestamp, severity (CHECK), resolved | CHECK IN ('LOW','MEDIUM','HIGH','CRITICAL') |

Then add the 3 indexes for query performance.

**Test your SQL:**
```bash
# Apply the schema to your local PostgreSQL:
psql -U visionstream -d visionstream -f db/schema.sql
```

### Step 2 — Write the Complex Query (`db/schema.sql`)

At the bottom of `db/schema.sql`, write the 24-hour geo-grid alert frequency query.
Then copy it into `app/services/alert_service.get_alert_stats()` as a `text()` query.

### Step 3 — Implement ORM Models (`app/models/`)

| File | Class | What to implement |
|------|-------|-------------------|
| `app/models/device.py` | `Device` | All columns, 2 relationships (to TelemetryLog, Alert) |
| `app/models/telemetry.py` | `TelemetryLog` | All columns, relationship back to Device |
| `app/models/alert.py` | `Alert` | All columns, CHECK constraint in `__table_args__`, relationship |

### Step 4 — Database Connection (`app/database.py`)

Implement:
- `engine` with connection pooling (pool_size=10, max_overflow=20)
- `SessionLocal` factory
- `get_db()` generator (yield + finally close)

### Step 5 — Seed the Database (`scripts/seed_geolife_data.py`)

Implement the seed script to populate **all three tables** in this exact order:

```
1. devices      ← Insert first (parent table, no foreign key dependencies)
2. telemetry_logs ← Insert second (depends on devices.device_id)
3. alerts       ← Insert third (depends on devices.device_id)
```

Your seed script must include these functions:

| Function | Purpose |
|----------|---------|
| `insert_seed_device()` | Insert one placeholder device into the `devices` table |
| `parse_plt_file()` | Parse one .plt file, yield dicts with **random** sensor data |
| `batch_insert_telemetry()` | Bulk insert telemetry_logs records |
| `generate_alerts_from_telemetry()` | Filter telemetry records with alerts, convert to alert dicts |
| `batch_insert_alerts()` | Bulk insert alerts records |
| `seed_database()` | Orchestrate the full import in the correct order |

**Important requirements:**
- `obstacle_distance_cm` and `battery_level` must be random integers, NOT `None`
- About 5% of telemetry records should have a non-`NONE` alert type (to generate alerts)
- Use `random.randint()` and `random.choice()` to generate realistic values
- Use `bulk_insert_mappings()` for batch inserts (never insert one row at a time)

```bash
# Run the seed script:
python scripts/seed_geolife_data.py

# Verify rows were inserted:
# psql> SELECT COUNT(*) FROM telemetry_logs;
# psql> SELECT COUNT(*) FROM alerts;
# psql> SELECT * FROM devices;
```

### Step 6 — Implement `alert_service.get_alert_stats()`

In `app/services/alert_service.py`, implement `get_alert_stats()` using the SQL query
you wrote in Step 2. Use `db.execute(text(...), {...})` to run the parameterized query.

---

## ✅ Lesson 2 Self-Assessment Checklist

- [ ] `db/schema.sql` creates all 3 tables without errors in psql
- [ ] All 3 indexes are created in `db/schema.sql`
- [ ] The complex geo-grid aggregation query returns results on seeded data
- [ ] `Device`, `TelemetryLog`, `Alert` ORM models match the SQL schema
- [ ] ORM relationships work: `device.telemetry_logs` and `device.alerts` are accessible
- [ ] `app/database.py` `get_db()` is implemented (yields session, closes in finally)
- [ ] `python scripts/seed_geolife_data.py` imports at least 10,000 telemetry rows
- [ ] `python scripts/seed_geolife_data.py` generates at least some alert rows
- [ ] `SELECT COUNT(*) FROM telemetry_logs;` shows rows in psql
- [ ] `SELECT COUNT(*) FROM alerts;` shows rows in psql
- [ ] `obstacle_distance_cm` and `battery_level` have non-NULL values in all rows
- [ ] `GET /alerts/stats` endpoint returns data when alerts exist
- [ ] GitHub commit pushed: `feat(db): complete Lesson 2 schema and seed script`

---

**When all boxes are checked → you are ready for Lesson 3.**